In [1]:
import sys
import json
from tqdm import tqdm
from openai import OpenAI 
import os
from functools import reduce
from typing import Dict

# TO CHANGE
BASEDIR = "/home/dzigen/Desktop/PersonalAI/Personal-AI"
# TO CHNAGE

sys.path.insert(0, BASEDIR)

from src.memorize_pipeline import MemPipeline
from src.memorize_pipeline.extractor.LLMExtractor import LLMExtractor
from src.memorize_pipeline.updator.LLMUpdator import LLMUpdator
from src.llm_agent import AgentConnector
from src.utils.data_structs import TripletCreator, NodeCreator, Relation, NODES_TYPES_MAP, RELATIONS_TYPES_MAP
from src.llm_agent.agent_model import SYSTEM_PROMPT

from src.neo4j_functions import Neo4jConnection
from src.embedding_functions import EmbeddingsDatabaseConnection, EmbeddingsDatabaseConnectionConfig, VectorDBConnectionConfig
from src.knowledge_graph_model import KnowledgeGraphModel

DATASET_PATH = '../data/Augment_DiaASQ.json'

ModuleNotFoundError: No module named 'openai'

In [2]:
class OpenAIAgent:

    def __init__(self, api_key, model: str = 'gpt-4o-mini') -> None:
        self.model = model
        self.client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", api_key))
        self.system_prompt = SYSTEM_PROMPT

    def generate(self, user_prompt: str, assistant_prompt: str = None, 
                 system_prompt: str = None, gen_strategy: Dict = None) -> str:
        """Метод для генерации ответов на текстовые запросы с помощью llm-агента.

        Args:
            user_prompt (str): Запрос для llm-агента.
            assistant_prompt (str, optional): Дополнительная к user_prompt-запросу информация, 
                                                которая может быть использована llm-агентом при генерации ответа. Defaults to None.
            gen_strategy (Dict, optional): Стретегия генерации текстовой последовательности для llm-агента. Defaults to None.

        Returns:
            str: Текстовая последовательность, сгенерированная llm-агентом.
        """
        messages = [
            {"role": "system", "content": system_prompt if system_prompt is not None else self.system_prompt},
            {"role": "user","content": user_prompt}]

        if assistant_prompt is not None:
            messages.insert(1, {"role": "assistant", "content": assistant_prompt})

        completion = self.client.chat.completions.create(
            model=self.model, messages=messages)

        return completion.choices[0].message.content

In [3]:
API_KEY = "'sk-861mINAavom2SSBqgrI82D4thMOfqT37knCof2o0H0T3BlbkFJ2gdVXJuVjNesNNP2aeUwPoBpZP3a3R1gn1kqv97CsA'"
agent = OpenAIAgent(api_key=API_KEY)

In [9]:
agent.generate("Сколько будет 2+2?")

'2 + 2 будет 4.'

### Extract

In [4]:
with open(DATASET_PATH, 'r', encoding='utf-8') as fd:
    data = json.loads(fd.read())

In [5]:
raw_texts = list(map(lambda v: v['text_dialog'], data['data']))
print(len(raw_texts))

In [4]:
#agent = AgentConnector.open()
#agent.generate("Сколько будет 2 + 2?")

In [7]:
extractor = LLMExtractor(agent_conn=agent)

In [8]:
extracted_triplets = []

In [9]:
for i in tqdm(range(len(raw_texts))):
    out = extractor.extract(raw_texts[i])
    extracted_triplets.append(out)

100%|██████████| 3483/3483 [6:25:49<00:00,  6.65s/it]  


In [10]:
print(sum(list(map(len, extracted_triplets))))
with open("tmp_extracted_openai_gpt4omini_triplets.json", 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(extracted_triplets, ensure_ascii=False))

### Update

In [4]:
kg_model = KnowledgeGraphModel(
    graph_db=Neo4jConnection(uri="bolt://localhost:7687", user="neo4j", pwd="password", db_name="diaasq2"),
    embeddings_db=EmbeddingsDatabaseConnection(EmbeddingsDatabaseConnectionConfig(
        nodes_db_config=VectorDBConnectionConfig(
            '../../data/graph_structures/vectorized_nodes/v10/densedb', 'vectorized_nodes', is_exist=False, need_to_clear=False
        ),
        triplets_db_config=VectorDBConnectionConfig(
            '../../data/graph_structures/vectorized_triplets/v6/densedb', 'vectorized_triplets', is_exist=False, need_to_clear=False
        )
    ))
)

No sentence-transformers model found with name ../models/intfloat/multilingual-e5-small. Creating a new one with MEAN pooling.


In [2]:
extracted_triplets = json.loads(open("../data/tmp_new_graph_extracted/tmp_extracted_openai_gpt4omini_triplets.json", 'r', encoding='utf-8').read())
print(extracted_triplets)

# adding time

283268


In [ ]:
extracted_triplets = reduce(lambda acc, v: acc + v, extracted_triplets, [])
print(len(extracted_triplets))

In [8]:
formated_triplets = []
for raw_triplet in tqdm(extracted_triplets):
    formated_triplets.append(TripletCreator.create(
        NodeCreator.create(name=raw_triplet[0]['name'], type=NODES_TYPES_MAP[raw_triplet[0]['type']], add_stringified_node=False),
        Relation(name=raw_triplet[1]['name'], type=RELATIONS_TYPES_MAP[raw_triplet[1]['prop']['type']]),
        NodeCreator.create(name=raw_triplet[2]['name'], type=NODES_TYPES_MAP[raw_triplet[2]['type']], add_stringified_node=False)
    ))

kg_model.graph_db.create_triplets(formated_triplets)

100%|██████████| 38714/38714 [2:28:26<00:00,  4.35it/s]  


In [ ]:
kg_model.graph_db.create_triplets(extracted_triplets)

In [ ]:
kg_model.embeddings_db.add_triplets(prepared_triplets)

In [ ]:
kg_model.graph_db.close()